# Predicting Microsatellite Instability from Tumor Gene Expression
### Metric: Macro F1-score

## Contexto biológico
- **MSI-H**: alta inestabilidad microsatelital — defecto en sistema de reparación de DNA, responde bien a inmunoterapia
- **MSI-L**: inestabilidad leve
- **MSS**: estable — clase mayoritaria

## Desafío técnico
- ~33,379 genes (features) con pocas centenas de muestras → p >> n extremo
- Datos ya log-transformados (log1p aplicado upstream)
- ~36% zeros (sparsidad moderada)
- MSS dominante → class weights necesarios

## Pipeline
1. Selección de genes por varianza (eliminar genes sin señal)
2. ANOVA F-score → top K genes discriminativos
3. PCA → reducción a ~50-100 componentes
4. Ensemble: SVM-RBF + LogisticRegression(ElasticNet) + LGB regularizado
5. Meta-learner sobre probabilidades OOF

In [1]:
import subprocess, sys
def pip(pkg): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

(None, None)

In [2]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from scipy.optimize import minimize

import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.feature_selection import f_classif, mutual_info_classif
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight

In [3]:
PLATFORM_DATA_DIR = Path('dataset/public')
if PLATFORM_DATA_DIR.exists():
    TRAIN_PATH  = PLATFORM_DATA_DIR / 'train_short.csv'
    TEST_PATH   = PLATFORM_DATA_DIR / 'test.csv'
    SUBMIT_PATH = Path('working/submission.csv')
    Path('working').mkdir(exist_ok=True)
else:
    from google.colab import files
    files.upload()
    TRAIN_PATH  = Path('train_short.csv')
    TEST_PATH   = Path('test.csv')
    SUBMIT_PATH = Path('submission.csv')

print('Loading data...')
train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
print(f'Train: {train.shape} | Test: {test.shape}')

Saving train.csv to train.csv
Saving test.csv to test.csv
Saving sample_submission.csv to sample_submission.csv
Loading data...
Train: (348, 33381) | Test: (87, 33380)


In [46]:
TARGET    = 'MSI_status'
CLASSES   = ['MSI-H', 'MSI-L', 'MSS']
GENE_COLS = [c for c in train.columns if c not in ['id', TARGET]]

SEED    = 42
MODE    = 'precision'  # 'quickrun' | 'precision'
N_FOLDS = 3 if MODE == 'quickrun' else 8
SEEDS   = [42] if MODE == 'quickrun' else [42, 7, 123]

np.random.seed(SEED)

# Encode labels
le = LabelEncoder().fit(CLASSES)
y  = le.transform(train[TARGET].values)  # MSI-H=0, MSI-L=1, MSS=2

print(f'Genes: {len(GENE_COLS)}')
print(f'Class distribution:')
for c, yi in zip(le.classes_, range(3)):
    print(f'  {c}: {(y==yi).sum()} ({(y==yi).mean()*100:.1f}%)')
print(f'\nMode: {MODE} | Folds: {N_FOLDS} | Seeds: {SEEDS}')

Genes: 33379
Class distribution:
  MSI-H: 55 (15.8%)
  MSI-L: 58 (16.7%)
  MSS: 235 (67.5%)

Mode: precision | Folds: 8 | Seeds: [42, 7, 123]


In [47]:
# ── Macro F1 helper ────────────────────────────────────────────────────────────
def macro_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, average='macro')

def per_class_f1(y_true, y_pred):
    scores = f1_score(y_true, y_pred, average=None, labels=[0,1,2])
    for c, s in zip(CLASSES, scores):
        print(f'  {c}: {s:.4f}')
    return scores

In [110]:
# ── Reducción de dimensionalidad ───────────────────────────────────────────────
X_raw    = train[GENE_COLS].values.astype(np.float32)
gene_var = X_raw.var(axis=0)
# ── Variance filter con codo natural ──────────────────────────────────────────
gene_var = X_raw.var(axis=0)
var_sorted = np.sort(gene_var)[::-1]  # de mayor a menor

# Encontrar el codo: punto donde la caída de varianza es más brusca
# Usar segunda derivada — el máximo indica el cambio más abrupto
diffs  = np.diff(var_sorted)           # primera derivada (caída)
diffs2 = np.diff(diffs)                # segunda derivada (cambio en la caída)

n = len(var_sorted)
search_range = slice(int(n*0.01), int(n*0.80))  # buscar entre percentil 1 y 80
elbow_idx = np.argmax(np.abs(diffs2[search_range])) + int(n*0.01)

var_threshold = var_sorted[elbow_idx]
high_var_mask = gene_var > 0

print(f'Total genes: {len(gene_var)}')
print(f'Elbow at index: {elbow_idx} ({elbow_idx/n*100:.1f}% percentile)')
print(f'Variance threshold: {var_threshold:.6f}')
print(f'Genes retained: {high_var_mask.sum()} ({high_var_mask.mean()*100:.1f}%)')

# Verificación visual de la distribución
percentiles = [i for i in range(11)]
print(f'\nDistribución de varianza:')
for p in percentiles:
    v = np.percentile(gene_var, p)
    marker = ' ← CODO' if abs(v - var_threshold) < var_threshold * 0.1 else ''
    print(f'  P{p:>3}: {v:.6f}{marker}')

X_hv     = X_raw[:, high_var_mask]

X_test_raw = test[GENE_COLS].values.astype(np.float32)
X_test_hv  = X_test_raw[:, high_var_mask]

print(f'Genes after variance filter: {X_hv.shape[1]} (de {X_raw.shape[1]})')

# Parámetros que usa fit_reduce() en cada fold
K_GENES = 10000
N_PCS   = min(len(X_hv), 150)
print(f'K_GENES={K_GENES} | N_PCS={N_PCS}')

Total genes: 33379
Elbow at index: 414 (1.2% percentile)
Variance threshold: 3.029488
Genes retained: 32673 (97.9%)

Distribución de varianza:
  P  0: 0.000000
  P  1: 0.000000
  P  2: 0.000000
  P  3: 0.001377
  P  4: 0.001377
  P  5: 0.002745
  P  6: 0.004106
  P  7: 0.005459
  P  8: 0.006804
  P  9: 0.008220
  P 10: 0.010199
Genes after variance filter: 32673 (de 33379)
K_GENES=10000 | N_PCS=150


In [102]:
# ── Class weights ──────────────────────────────────────────────────────────────
cw     = compute_class_weight('balanced', classes=np.array([0,1,2]), y=y)
cw_dict = {0: cw[0], 1: cw[1], 2: cw[2]}
print(f'Class weights: MSI-H={cw[0]:.2f} MSI-L={cw[1]:.2f} MSS={cw[2]:.2f}')

Class weights: MSI-H=2.11 MSI-L=2.00 MSS=0.49


In [103]:
# ── Selección de genes ÚNICA (fuera del CV) ────────────────────────────────────
print('Computing gene selection (ANOVA + MI)...')
f_scores, _  = f_classif(X_hv, y)
mi_scores    = mutual_info_classif(X_hv, y, random_state=SEED, n_jobs=-1)
f_scores     = np.nan_to_num(f_scores)
mi_scores    = np.nan_to_num(mi_scores)

k_half   = K_GENES // 2
top_f    = set(np.argsort(f_scores)[::-1][:k_half])
top_mi   = set(np.argsort(mi_scores)[::-1][:k_half])
mask_l   = y == 1
mask_oth = y != 1
diff_l   = np.abs(X_hv[mask_l].mean(axis=0) - X_hv[mask_oth].mean(axis=0))
top_msil = set(np.argsort(diff_l)[::-1][:K_GENES // 4])

top_idx  = np.array(sorted(top_f | top_mi | top_msil))
print(f'Selected genes: {len(top_idx)} (ANOVA={len(top_f)} MI={len(top_mi)} MSI-L={len(top_msil)})')

# Library norm + selección aplicada una vez
def lib_norm(X):
    # Library size normalization
    row_sums = X.sum(axis=1, keepdims=True)
    X_norm = X / (row_sums + 1e-9) * np.median(row_sums)
    # Z-score por muestra — captura patrones relativos
    X_znorm = (X_norm - X_norm.mean(axis=1, keepdims=True)) / (X_norm.std(axis=1, keepdims=True) + 1e-9)
    return X_znorm

X_sel   = lib_norm(X_hv)[:, top_idx]
Xte_sel = lib_norm(X_test_hv)[:, top_idx]
print(f'X_sel: {X_sel.shape} | Xte_sel: {Xte_sel.shape}')

Computing gene selection (ANOVA + MI)...
Selected genes: 8847 (ANOVA=5000 MI=5000 MSI-L=2500)
X_sel: (348, 8847) | Xte_sel: (87, 8847)


In [111]:
# ── Interacciones multiplicativas de top genes MSI-L ─────────────────────────
# Calcular sobre X_hv antes de selección
top_msil_global = np.argsort(
    np.abs(X_hv[y==1].mean(axis=0) - X_hv[y!=1].mean(axis=0))
)[::-1][:20]  # top 20 genes MSI-L específicos

prod_features_tr = []
prod_features_te = []
for i in range(len(top_msil_global)):
    for j in range(i+1, len(top_msil_global)):
        gi, gj = top_msil_global[i], top_msil_global[j]
        prod_features_tr.append(X_hv[:, gi] * X_hv[:, gj])
        prod_features_te.append(X_test_hv[:, gi] * X_test_hv[:, gj])

prod_tr = np.column_stack(prod_features_tr).astype(np.float32)  # (348, 190)
prod_te = np.column_stack(prod_features_te).astype(np.float32)

sc_prod    = StandardScaler().fit(prod_tr)
prod_tr_s  = sc_prod.transform(prod_tr).astype(np.float32)
prod_te_s  = sc_prod.transform(prod_te).astype(np.float32)

print(f'Producto features: {prod_tr_s.shape}')

# PCA entrenado una sola vez sobre train + test combinados
print('Fitting PCA on train + test...')
sc_global  = StandardScaler().fit(np.vstack([X_sel, Xte_sel]))
X_pca_all  = sc_global.transform(np.vstack([X_sel, Xte_sel]))
pca_global = PCA(n_components=N_PCS, random_state=SEED).fit(X_pca_all)

X_pca_tr = pca_global.transform(sc_global.transform(X_sel)).astype(np.float32)
X_pca_te = pca_global.transform(sc_global.transform(Xte_sel)).astype(np.float32)
print(f'Feature matrix final: train={X_pca_tr.shape} | test={X_pca_te.shape}')

# Concatenar al feature matrix final
X_pca_tr = np.hstack([X_pca_tr, prod_tr_s])
X_pca_te = np.hstack([X_pca_te, prod_te_s])

cumvar = np.cumsum(pca_global.explained_variance_ratio_)
print(f'Variance explained: PC10={cumvar[9]*100:.1f}% | PC{N_PCS}={cumvar[-1]*100:.1f}%')

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

def fit_reduce(tr_idx, va_idx):
    Xtr = X_sel[tr_idx]
    Xva = X_sel[va_idx]
    Xte = Xte_sel

    sc  = StandardScaler().fit(Xtr)
    Xtr_s = sc.transform(Xtr)
    Xva_s = sc.transform(Xva)
    Xte_s = sc.transform(Xte)

    # PCA global ya calculado — solo proyectar
    Xtr_pca = pca_global.transform(Xtr_s).astype(np.float32)
    Xva_pca = pca_global.transform(Xva_s).astype(np.float32)
    Xte_pca = pca_global.transform(Xte_s).astype(np.float32)

    # LDA dentro del fold — usa labels solo del train fold
    lda = LinearDiscriminantAnalysis()
    lda.fit(Xtr_pca, y[tr_idx])
    sc_lda = StandardScaler().fit(lda.transform(Xtr_pca))

    Xtr_lda = sc_lda.transform(lda.transform(Xtr_pca)).astype(np.float32)
    Xva_lda = sc_lda.transform(lda.transform(Xva_pca)).astype(np.float32)
    Xte_lda = sc_lda.transform(lda.transform(Xte_pca)).astype(np.float32)

    return (np.hstack([Xtr_pca, Xtr_lda, stats_tr_s[tr_idx]]),
            np.hstack([Xva_pca, Xva_lda, stats_te_s if False else stats_tr_s[va_idx]]),
            np.hstack([Xte_pca, Xte_lda, stats_te_s]))

Producto features: (348, 190)
Fitting PCA on train + test...
Feature matrix final: train=(348, 150) | test=(87, 150)
Variance explained: PC10=42.7% | PC150=80.3%


In [112]:
# ── Sample-level statistical features ─────────────────────────────────────────
# Características de la distribución global de expresión por muestra
# Se concatenan a los PCs para enriquecer la representación

def sample_stats(X_raw):
    """X_raw: array original sin normalizar (o lib_norm) — (n_samples, n_genes)"""
    mean_  = X_raw.mean(axis=1)
    std_   = X_raw.std(axis=1)
    return np.column_stack([
        mean_,                                                    # expresión media
        std_,                                                     # dispersión
        pd.DataFrame(X_raw).skew(axis=1).values,                 # asimetría
        pd.DataFrame(X_raw).kurt(axis=1).values,                 # curtosis
        (X_raw == 0).mean(axis=1),                               # fracción zeros
        (X_raw > 0).sum(axis=1),                                  # genes activos
        (X_raw > mean_.reshape(-1,1) + std_.reshape(-1,1)).sum(axis=1),  # genes muy activos
        np.percentile(X_raw, 25, axis=1),                        # Q1
        np.percentile(X_raw, 75, axis=1),                        # Q3
        np.percentile(X_raw, 75, axis=1) - np.percentile(X_raw, 25, axis=1),  # IQR
        np.percentile(X_raw, 90, axis=1),                        # P90
        np.percentile(X_raw, 99, axis=1),                        # P99
        X_raw.max(axis=1) - X_raw.min(axis=1),                   # rango
        std_ / (mean_ + 1e-9),                                   # coef. variación
    ]).astype(np.float32)

# Calcular sobre X_hv (antes de selección) para capturar distribución completa
stats_tr = sample_stats(X_hv)
stats_te = sample_stats(X_test_hv)

# Escalar las stats
sc_stats   = StandardScaler().fit(stats_tr)
stats_tr_s = sc_stats.transform(stats_tr).astype(np.float32)
stats_te_s = sc_stats.transform(stats_te).astype(np.float32)

# Concatenar PCA + stats
X_pca_tr = np.hstack([X_pca_tr, stats_tr_s])
X_pca_te = np.hstack([X_pca_te, stats_te_s])

print(f'Feature matrix con stats: train={X_pca_tr.shape} | test={X_pca_te.shape}')
print(f'(PCs={N_PCS} + stats=14 = {N_PCS+14} features)')

Feature matrix con stats: train=(348, 354) | test=(87, 354)
(PCs=150 + stats=14 = 164 features)


In [113]:
# ── Genes directos desde PCs más discriminativos ──────────────────────────────
# Paso 1: identificar qué PCs tienen señal real (F > umbral)
from scipy.stats import f_oneway

pc_signal = []
for i in range(X_pca_tr.shape[1]):
    f_val, p_val = f_oneway(X_pca_tr[y==0, i],
                             X_pca_tr[y==1, i],
                             X_pca_tr[y==2, i])
    pc_signal.append((i, f_val, p_val))

# Solo PCs con F significativo Y dentro del rango de pca_global
n_pca_components = pca_global.components_.shape[0]
signal_pcs = [(i, f) for i, f, p in pc_signal
              if p < 0.05 and i < n_pca_components]
signal_pcs.sort(key=lambda x: -x[1])
print(f'PCs con señal significativa (en rango PCA): {len(signal_pcs)}')
for i, f in signal_pcs[:5]:
    print(f'  PC{i+1}: F={f:.2f}')
print(f'PCs con señal significativa: {len(signal_pcs)}')
for i, f in signal_pcs[:5]:
    print(f'  PC{i+1}: F={f:.2f}')

# Paso 2: extraer top genes por loading en cada PC significativo
N_GENES_PER_PC = 5  # cuántos genes extraer por PC
TOP_N_PCS      = 3  # cuántos PCs considerar

selected_direct = set()
for pc_idx, f_val in signal_pcs[:TOP_N_PCS]:
    loadings  = np.abs(pca_global.components_[pc_idx])
    top_genes = np.argsort(loadings)[::-1][:N_GENES_PER_PC]
    # top_genes son índices en X_sel → mapear a X_hv
    orig_idx  = top_idx[top_genes]  # top_idx es el mapping sel→hv
    selected_direct.update(orig_idx.tolist())
    print(f'  PC{pc_idx+1} (F={f_val:.1f}): genes {orig_idx.tolist()}')

selected_direct = sorted(selected_direct)
print(f'\nTotal genes directos seleccionados: {len(selected_direct)}')

# Paso 3: agregar como features directas
direct_tr = np.column_stack([X_hv[:, g] for g in selected_direct]).astype(np.float32)
direct_te = np.column_stack([X_test_hv[:, g] for g in selected_direct]).astype(np.float32)

sc_direct   = StandardScaler().fit(direct_tr)
direct_tr_s = sc_direct.transform(direct_tr).astype(np.float32)
direct_te_s = sc_direct.transform(direct_te).astype(np.float32)

X_pca_tr = np.hstack([X_pca_tr, direct_tr_s])
X_pca_te = np.hstack([X_pca_te, direct_te_s])
print(f'Feature matrix con genes directos: train={X_pca_tr.shape} | test={X_pca_te.shape}')

PCs con señal significativa (en rango PCA): 20
  PC2: F=127.29
  PC3: F=17.39
  PC4: F=9.31
  PC5: F=8.58
  PC80: F=8.37
PCs con señal significativa: 20
  PC2: F=127.29
  PC3: F=17.39
  PC4: F=9.31
  PC5: F=8.58
  PC80: F=8.37
  PC2 (F=127.3): genes [5275, 6825, 18952, 7023, 31053]
  PC3 (F=17.4): genes [3305, 17113, 13333, 15065, 28590]
  PC4 (F=9.3): genes [16831, 7002, 22941, 4461, 26002]

Total genes directos seleccionados: 15
Feature matrix con genes directos: train=(348, 369) | test=(87, 369)


In [114]:
# ════════════════════════════════════════════════════════════════════════
# MODELO: Neural Network multi-clase
# ════════════════════════════════════════════════════════════════════════
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

class ResBlock(nn.Module):
    def __init__(self, dim, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim), nn.BatchNorm1d(dim), nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim), nn.BatchNorm1d(dim),
        )
        self.act = nn.SiLU()
    def forward(self, x):
        return self.act(x + self.net(x))

class MSINet(nn.Module):
    def __init__(self, n_in, hidden=256, n_blocks=4, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, hidden), nn.BatchNorm1d(hidden), nn.SiLU(),
            nn.Dropout(dropout),
            *[ResBlock(hidden, dropout) for _ in range(n_blocks)],
            nn.Linear(hidden, 64), nn.SiLU(),
            nn.Linear(64, 3)
        )
    def forward(self, x):
        return self.net(x)

print('\n=== Neural Network ===')
oof_nn  = np.zeros((len(X_hv), 3))
test_nn = np.zeros((len(X_test_hv), 3))

cw_tensor = torch.tensor(cw, dtype=torch.float32).to(DEVICE)
epochs    = 200 if MODE == 'quickrun' else 500

for seed in SEEDS:
    torch.manual_seed(seed)
    skf   = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    oof_s = np.zeros((len(X_hv), 3))
    te_s  = np.zeros((len(X_test_hv), 3))

    for tr, va in skf.split(X_pca_tr, y):
        Xtr, Xva, Xte = fit_reduce(tr, va)

        Xtr_t = torch.tensor(Xtr, dtype=torch.float32)
        Xva_t = torch.tensor(Xva, dtype=torch.float32)
        Xte_t = torch.tensor(Xte, dtype=torch.float32)
        ytr_t = torch.tensor(y[tr], dtype=torch.long)

        loader = DataLoader(TensorDataset(Xtr_t, ytr_t),
                            batch_size=32, shuffle=True)

        model = MSINet(Xtr.shape[1]).to(DEVICE)
        opt   = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-3)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
        crit  = nn.CrossEntropyLoss(weight=cw_tensor, label_smoothing=0.1)

        best_f1, best_state, no_imp = -1, None, 0
        for epoch in range(epochs):
            model.train()
            for Xb, yb in loader:
                Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
                loss = crit(model(Xb), yb)
                opt.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
            sched.step()
            model.eval()
            with torch.no_grad():
                pva  = torch.softmax(model(Xva_t.to(DEVICE)), dim=1).cpu().numpy()
                vf1  = macro_f1(y[va], np.argmax(pva, axis=1))
            if vf1 > best_f1:
                best_f1    = vf1
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
                no_imp     = 0
            else:
                no_imp += 1
            if no_imp >= 50:
                break

        model.load_state_dict(best_state)
        model.eval()
        with torch.no_grad():
            oof_s[va] = torch.softmax(model(Xva_t.to(DEVICE)), dim=1).cpu().numpy()
            te_s     += torch.softmax(model(Xte_t.to(DEVICE)), dim=1).cpu().numpy() / N_FOLDS

    oof_nn  += oof_s  / len(SEEDS)
    test_nn += te_s   / len(SEEDS)
    print(f'  Seed {seed}: {macro_f1(y, np.argmax(oof_s,1)):.4f}')

print(f'NN OOF F1: {macro_f1(y, np.argmax(oof_nn,1)):.4f}')


=== Neural Network ===
  Seed 42: 0.7535
  Seed 7: 0.7855
  Seed 123: 0.7645
NN OOF F1: 0.7129


In [115]:
# ── Threshold optimizer ─────────────────────────────────────
def apply_thresholds(proba, thresholds):
    return np.argmax(proba - np.array(thresholds), axis=1)

best_f1, best_thresh = -1, [0.0, 0.0, 0.0]
for t_msi_l in np.arange(-0.3, 0.1, 0.005):
    for t_mss in np.arange(-0.1, 0.15, 0.005):
        thresh = [0.0, t_msi_l, t_mss]
        score  = macro_f1(y, apply_thresholds(oof_nn, thresh))
        if score > best_f1:
            best_f1, best_thresh = score, thresh

print(f'Best: MSI-L={best_thresh[1]:.3f} MSS={best_thresh[2]:.3f} → F1={best_f1:.4f}')
print('Per-class:')
_ = per_class_f1(y, apply_thresholds(oof_nn, best_thresh))
opt_thresh = best_thresh

Best: MSI-L=-0.300 MSS=0.035 → F1=0.7793
Per-class:
  MSI-H: 0.9725
  MSI-L: 0.5077
  MSS: 0.8578


In [116]:
# ── Submission ─────────────────────────────────────────────────────────────────
final_preds = apply_thresholds(test_nn, opt_thresh)
final_labels = le.inverse_transform(final_preds)

submission = pd.DataFrame({'id': test['id'], 'prediction': final_labels})
submission.to_csv(SUBMIT_PATH, index=False)
print(f'Saved: {SUBMIT_PATH}')
print(f'Prediction distribution:')
print(submission['prediction'].value_counts())
print(submission.head())

Saved: submission.csv
Prediction distribution:
prediction
MSS      54
MSI-L    18
MSI-H    15
Name: count, dtype: int64
   id prediction
0   0      MSI-H
1   7        MSS
2   8      MSI-L
3  13      MSI-H
4  23        MSS
